# 11 — Comparar el troceo con límite duro con nomic-embed-text en Qdrant

Medimos si `trocear_con_limite` (ningún chunk por encima de 1500 caracteres salvo un bloque de código indivisible, con el contexto repetido en cada trozo) mejora la recuperación frente a `trocear_por_estructura` (la configuración final de hoy). Se fija el modelo (`nomic-embed-text`, Ollama), la base (Qdrant, coseno) y k=4. Cambian el troceo (`actual` y `limitado`, ambos 1500/200), la ruta de títulos en el texto embebido y el reranker (sin reranker, o 20 candidatos reordenados con `cross-encoder/ms-marco-MiniLM-L-6-v2` hasta 4). Son 8 configuraciones. Se evalúa con el golden set de `docs/GOLDEN_SET_PYTHON_DOCS.md` y las evidencias de varias redacciones de `docs/golden_set_evidencias.json`.

Necesita Qdrant en `localhost:6333` (`docker compose up -d`), Ollama con `nomic-embed-text` (`ollama pull nomic-embed-text`) y la colección `pydocs_nomic_est1500_m200_sinruta`, creada con `notebooks/07_comparar_troceo_estructura.ipynb`. Esa colección no se borra nunca: la usa `scripts/consultar_rag.py`. Se crean o recrean solo `pydocs_nomic_est1500_m200_ruta`, `pydocs_nomic_lim1500_m200_ruta` y `pydocs_nomic_lim1500_m200_sinruta` (tres indexaciones de varios minutos).

## 1. Dependencias

Ya están declaradas en `pyproject.toml` (se instalan con `uv sync`). Si falta alguna, se añade con `uv add <paquete>`.

## 2. Imports y configuración

`CONFIGURACIONES` es el único sitio donde se declaran las configuraciones; el resto de celdas las recorre. Cada colección se indexa una vez y sirve a las dos variantes de reranker.

In [1]:
import sys
import time
from pathlib import Path

import ollama
import pandas as pd
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

# Las funciones puras viven en scripts/ (la raíz del repo es el directorio padre)
RAIZ = Path.cwd() if (Path.cwd() / "scripts").exists() else Path.cwd().parent
sys.path.insert(0, str(RAIZ / "scripts"))
from comparar_embeddings import (
    MODELOS, calcular_metricas, es_bloque_de_codigo, evidencias_fuera_de_chunks, fuentes_ausentes,
    leer_evidencias, leer_golden_set, preparar_documento, preparar_query,
    rango_primer_acierto, rango_primer_acierto_evidencia, reordenar_por_score, texto_a_embeber,
    trocear_con_limite, trocear_por_estructura,
)

CARPETA_CORPUS = RAIZ / "data" / "python-docs-3.15" / "python-3.15-docs-md-clean"
RUTA_GOLDEN = RAIZ / "docs" / "GOLDEN_SET_PYTHON_DOCS.md"
RUTA_EVIDENCIAS = RAIZ / "docs" / "golden_set_evidencias.json"
RUTA_CSV = RAIZ / "data" / "resultados" / "troceo_limitado.csv"
RUTA_CSV_PREGUNTAS = RAIZ / "data" / "resultados" / "troceo_limitado_por_pregunta.csv"
RUTA_CSV_ESTRUCTURA = RAIZ / "data" / "resultados" / "troceo_estructura.csv"
MODELO = "nomic"
MAXIMO = 1500
MINIMO = 200
CANDIDATOS = 20
K = 4
RERANKER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOTE_EMBEDDING = 64   # chunks por llamada al modelo
LOTE_INSERCION = 500  # puntos por inserción
COLECCION_PROTEGIDA = "pydocs_nomic_est1500_m200_sinruta"  # la usa scripts/consultar_rag.py: no se borra
CONFIGURACIONES = [
    {"troceo": troceo, "con_ruta": con_ruta, "reranker": reranker}
    for troceo in ("actual", "limitado")
    for con_ruta in (False, True)
    for reranker in (False, True)
]

## 3. Conexión con Qdrant y Ollama

Se comprueba antes de calcular nada. Si algo falta, la celda para con un mensaje que dice qué arrancar.

In [2]:
qdrant = QdrantClient("localhost", port=6333)
try:
    qdrant.get_collections()
except Exception as error:
    raise RuntimeError(
        "Qdrant no responde en localhost:6333. Arráncalo con `docker compose up -d` y vuelve a ejecutar."
    ) from error

modelo_ollama = MODELOS[MODELO]["nombre"]
try:
    ollama.show(modelo_ollama)
except ConnectionError as error:
    raise RuntimeError("Ollama no responde. Arráncalo (`ollama serve` o la app) y vuelve a ejecutar.") from error
except ollama.ResponseError as error:
    raise RuntimeError(
        f"Ollama no tiene el modelo {modelo_ollama}. Ejecuta `ollama pull {modelo_ollama}` y vuelve a ejecutar."
    ) from error

if not qdrant.collection_exists(COLECCION_PROTEGIDA):
    raise RuntimeError(
        f"Falta {COLECCION_PROTEGIDA}. Ejecuta `notebooks/07_comparar_troceo_estructura.ipynb` para crearla. No se crea aquí."
    )

print("Qdrant:", [col.name for col in qdrant.get_collections().collections])
print("Ollama:", modelo_ollama, "disponible")

Qdrant: ['pydocs_nomic_est1500_m200_ruta', 'pydocs_nomic_est1500_m200_sinruta', 'pydocs_nomic_lim1500_m200_ruta', 'pydocs_nomic_lim1500_m200_sinruta']
Ollama: nomic-embed-text disponible


## 4. Preguntas y evidencias del golden set

Se leen de las mismas fuentes que los notebooks 07 a 10. Una fuente del golden set que no existe en el corpus se avisa con su ID y su ruta.

In [3]:
preguntas = leer_golden_set(RUTA_GOLDEN)
ausentes = fuentes_ausentes(preguntas, CARPETA_CORPUS)
for id_, ruta in ausentes:
    print(f"AVISO: la fuente {ruta} de {id_} no existe en el corpus")
print(len(preguntas), "preguntas:", pd.Series([p["categoria"] for p in preguntas]).value_counts().sort_index().to_dict())

evidencias = leer_evidencias(RUTA_EVIDENCIAS)

35 preguntas: {'A': 10, 'B': 10, 'C': 9, 'D': 3, 'E': 3}


## 5. Troceo por troceo

Se trocea el corpus una vez por troceo (`actual` = `trocear_por_estructura`, `limitado` = `trocear_con_limite`, ambos 1500/200). Se cuentan los chunks por encima de 1500 caracteres, cuántos de ellos no son un bloque de código (`es_bloque_de_codigo` sobre el fragmento sin contexto) y las evidencias que quedan partidas. Para cada fichero se comprueba que la concatenación de los fragmentos (`texto` sin `contexto`) reproduce el fichero; si no, la celda falla.

In [4]:
def trocear(troceo: str) -> list[dict]:
    if troceo == "actual":
        return [{**c, "contexto": ""} for c in trocear_por_estructura(CARPETA_CORPUS, MAXIMO, MINIMO)]
    return trocear_con_limite(CARPETA_CORPUS, MAXIMO, MINIMO)


def comprobar_reproduce_ficheros(chunks: list[dict]) -> None:
    """Por fichero, la concatenación de los fragmentos (sin el contexto repetido) es el fichero."""
    por_fuente: dict[str, list[str]] = {}
    for c in chunks:
        por_fuente.setdefault(c["fuente"], []).append(c["texto"][len(c["contexto"]):])
    for fuente, textos in por_fuente.items():
        original = (CARPETA_CORPUS / fuente).read_text(encoding="utf-8")
        assert "".join(textos) == original, f"El troceo no reproduce {fuente}"


chunks_por_troceo: dict[str, list[dict]] = {}
estadisticas: dict[str, dict] = {}
partidas: dict[str, list[tuple[str, str]]] = {}
for troceo in dict.fromkeys(c["troceo"] for c in CONFIGURACIONES):
    chunks = trocear(troceo)
    chunks_por_troceo[troceo] = chunks
    partidas[troceo] = evidencias_fuera_de_chunks(evidencias, chunks)
    longitudes = pd.Series([len(c["texto"]) for c in chunks])
    largos = [c for c in chunks if len(c["texto"]) > MAXIMO]
    no_codigo = [c for c in largos if not es_bloque_de_codigo(c["texto"][len(c["contexto"]):])]
    estadisticas[troceo] = {
        "chunks": len(chunks), "encima": len(largos), "encima_no_codigo": len(no_codigo),
        "medio": float(longitudes.mean()), "maximo": int(longitudes.max()),
    }
    print(f"{troceo}: {len(chunks)} chunks, tamaño medio {longitudes.mean():.0f}, máximo {longitudes.max()}, "
          f"{len(largos)} de más de {MAXIMO} ({len(no_codigo)} que no son un bloque de código), "
          f"{len(partidas[troceo])} evidencias partidas")
    for id_, fuente in partidas[troceo]:
        print(f"    partida: {id_} en {fuente}")
    comprobar_reproduce_ficheros(chunks)
    print("    reproduce los ficheros: sí")

actual: 7955 chunks, tamaño medio 1203, máximo 203072, 773 de más de 1500 (709 que no son un bloque de código), 0 evidencias partidas
    reproduce los ficheros: sí


limitado: 9143 chunks, tamaño medio 1079, máximo 22273, 69 de más de 1500 (0 que no son un bloque de código), 0 evidencias partidas
    reproduce los ficheros: sí


## 6. Embeddings e indexación por colección

Una colección por combinación de troceo y ruta; el texto embebido es `texto_a_embeber(chunk, con_ruta)` (con ruta antepone `ruta_titulos`; el `texto` del troceo limitado ya lleva el `contexto`). El payload guarda `fuente`, `texto`, `ruta_titulos` y `contexto`.

`pydocs_nomic_est1500_m200_sinruta` nunca se borra: se reutiliza si tiene tantos puntos como chunks del troceo actual y, si no, la celda para. Su tiempo de indexación se toma de `data/resultados/troceo_estructura.csv`. Las otras tres se recrean y se mide su tiempo.

In [5]:
def nombre_coleccion(c: dict) -> str:
    prefijo = "est" if c["troceo"] == "actual" else "lim"
    sufijo = "ruta" if c["con_ruta"] else "sinruta"
    return f"pydocs_{MODELO}_{prefijo}{MAXIMO}_m{MINIMO}_{sufijo}"


def embeber(textos: list[str], barra: bool = False, descripcion: str = "") -> list[list[float]]:
    """Embeddings de una lista de textos ya preparados, en lotes fijos."""
    vectores: list[list[float]] = []
    for i in tqdm(range(0, len(textos), LOTE_EMBEDDING), desc=descripcion, disable=not barra):
        vectores.extend(ollama.embed(model=modelo_ollama, input=textos[i:i + LOTE_EMBEDDING]).embeddings)
    return vectores


def indexar(nombre: str, chunks: list[dict], con_ruta: bool) -> float:
    assert nombre != COLECCION_PROTEGIDA, "No se recrea la colección protegida"
    inicio = time.perf_counter()
    vectores = embeber(
        [preparar_documento(MODELO, texto_a_embeber(c, con_ruta)) for c in chunks],
        barra=True, descripcion=nombre,
    )
    if qdrant.collection_exists(nombre):
        qdrant.delete_collection(nombre)
    qdrant.create_collection(
        nombre, vectors_config=VectorParams(size=len(vectores[0]), distance=Distance.COSINE)
    )
    for i in range(0, len(chunks), LOTE_INSERCION):
        qdrant.upsert(nombre, points=[
            PointStruct(id=c["id"], vector=v, payload={
                "fuente": c["fuente"], "texto": c["texto"],
                "ruta_titulos": c["ruta_titulos"], "contexto": c["contexto"],
            })
            for c, v in zip(chunks[i:i + LOTE_INSERCION], vectores[i:i + LOTE_INSERCION])
        ])
    return time.perf_counter() - inicio


def tiempo_coleccion_protegida(chunks: list[dict]) -> float:
    """Tiempo de indexación de la colección protegida (de troceo_estructura.csv). Para si no cuadra."""
    puntos = qdrant.count(COLECCION_PROTEGIDA, exact=True).count
    if puntos != len(chunks):
        raise RuntimeError(
            f"{COLECCION_PROTEGIDA} tiene {puntos} puntos y el troceo actual da {len(chunks)} chunks. "
            "No la borro: revísala."
        )
    if not RUTA_CSV_ESTRUCTURA.exists():
        raise RuntimeError(f"Falta {RUTA_CSV_ESTRUCTURA.relative_to(RAIZ)}.")
    tabla = pd.read_csv(RUTA_CSV_ESTRUCTURA)
    fila = tabla[tabla["coleccion"] == COLECCION_PROTEGIDA]
    if fila.empty:
        raise RuntimeError(f"{RUTA_CSV_ESTRUCTURA.relative_to(RAIZ)} no tiene la fila de {COLECCION_PROTEGIDA}.")
    return float(fila.iloc[0]["indexación (s)"])


colecciones = sorted(
    {(nombre_coleccion(c), c["troceo"], c["con_ruta"]) for c in CONFIGURACIONES},
    key=lambda t: (t[1], t[2]),
)
t_indexacion: dict[str, float] = {}
for nombre, troceo, con_ruta in colecciones:
    chunks = chunks_por_troceo[troceo]
    if nombre == COLECCION_PROTEGIDA:
        t_indexacion[nombre] = tiempo_coleccion_protegida(chunks)
        print(f"{nombre}: REUTILIZADA, {len(chunks)} puntos, indexación {t_indexacion[nombre]:.1f} s (de troceo_estructura.csv)")
        continue
    t_indexacion[nombre] = indexar(nombre, chunks, con_ruta)
    print(f"{nombre}: {len(chunks)} chunks en {t_indexacion[nombre]:.1f} s")

pydocs_nomic_est1500_m200_sinruta: REUTILIZADA, 7955 puntos, indexación 228.2 s (de troceo_estructura.csv)


pydocs_nomic_est1500_m200_ruta:   0%|          | 0/125 [00:00<?, ?it/s]

pydocs_nomic_est1500_m200_ruta: 7955 chunks en 129.8 s


pydocs_nomic_lim1500_m200_sinruta:   0%|          | 0/143 [00:00<?, ?it/s]

pydocs_nomic_lim1500_m200_sinruta: 9143 chunks en 146.3 s


pydocs_nomic_lim1500_m200_ruta:   0%|          | 0/143 [00:00<?, ?it/s]

pydocs_nomic_lim1500_m200_ruta: 9143 chunks en 146.9 s


## 7. Carga del reranker

Se carga una vez `cross-encoder/ms-marco-MiniLM-L-6-v2` (CPU) y se hace una llamada de calentamiento fuera de la medición de latencia.

In [6]:
try:
    reranker = CrossEncoder(RERANKER)
except Exception as error:
    raise RuntimeError(f"No se pudo cargar el reranker {RERANKER}: {error}") from error
reranker.predict([("calentamiento", "calentamiento")])
print(RERANKER, "cargado en", reranker.device)

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

cross-encoder/ms-marco-MiniLM-L-6-v2 cargado en cpu


## 8. Evaluación por colección

Para cada colección y cada pregunta (A-D y E) se hace una búsqueda de 20 candidatos (embedding más Qdrant) y se mide su latencia. Sin reranker, el resultado son los 4 primeros de esa búsqueda. Con reranker, se puntúan los pares (query sin prefijo, `texto` del chunk) con el cross-encoder y se queda con los 4 mejores (`reordenar_por_score`), midiendo su latencia aparte. Se guardan el rango por fuente y por evidencia (solo A-D).

In [7]:
def embeber_uno(texto: str) -> list[float]:
    return ollama.embed(model=modelo_ollama, input=[texto]).embeddings[0]


resultados = []
for nombre, troceo, con_ruta in colecciones:
    for p in tqdm(preguntas, desc=nombre):
        inicio = time.perf_counter()
        vector = embeber_uno(preparar_query(MODELO, p["query"]))
        res = qdrant.query_points(nombre, query=vector, limit=CANDIDATOS, with_payload=True).points
        busqueda_ms = (time.perf_counter() - inicio) * 1000
        candidatos = [{"fuente": r.payload["fuente"], "texto": r.payload["texto"]} for r in res]

        inicio = time.perf_counter()
        scores = [float(s) for s in reranker.predict([(p["query"], c["texto"]) for c in candidatos])]
        reordenados = reordenar_por_score(candidatos, scores, K)
        rerank_ms = (time.perf_counter() - inicio) * 1000

        for con_reranker, recuperados in ((False, candidatos[:K]), (True, reordenados)):
            fila = {
                "troceo": troceo, "con_ruta": con_ruta, "reranker": con_reranker, "coleccion": nombre,
                "id": p["id"], "categoria": p["categoria"],
                "busqueda_ms": busqueda_ms, "rerank_ms": rerank_ms if con_reranker else 0.0,
                "rango": None, "rango_evidencia": None,
            }
            if p["fuentes"]:
                fila["rango"] = rango_primer_acierto([r["fuente"] for r in recuperados], p["fuentes"])
                fila["rango_evidencia"] = rango_primer_acierto_evidencia(recuperados, evidencias[p["id"]])
            resultados.append(fila)

df = pd.DataFrame(resultados)
df["total_ms"] = df["busqueda_ms"] + df["rerank_ms"]
df_abcd = df[df["categoria"] != "E"]
print(len(df), "evaluaciones,", len(df_abcd), "con fuentes (A-D)")

pydocs_nomic_est1500_m200_sinruta:   0%|          | 0/35 [00:00<?, ?it/s]

pydocs_nomic_est1500_m200_ruta:   0%|          | 0/35 [00:00<?, ?it/s]

pydocs_nomic_lim1500_m200_sinruta:   0%|          | 0/35 [00:00<?, ?it/s]

pydocs_nomic_lim1500_m200_ruta:   0%|          | 0/35 [00:00<?, ?it/s]

280 evaluaciones, 256 con fuentes (A-D)


## 9. Métricas por configuración (preguntas A-D)

hit@1, hit@4 y MRR por fuente, y `ev_` por evidencia, con `calcular_metricas(rangos, 4)`.

In [8]:
CLAVE = ["troceo", "con_ruta", "reranker"]


def metricas(grupo: pd.DataFrame) -> pd.Series:
    """Métricas por fuente y por evidencia (prefijo `ev_`)."""
    filas = {}
    for columna, prefijo in (("rango", ""), ("rango_evidencia", "ev_")):
        rangos = [None if pd.isna(r) else int(r) for r in grupo[columna]]
        filas.update({prefijo + n: v for n, v in calcular_metricas(rangos, K).items()})
    return pd.Series(filas)


por_config = df_abcd.groupby(CLAVE).apply(metricas, include_groups=False)
por_config.round(3)

hit@1  hit@4    mrr  ev_hit@1  ev_hit@4  ev_mrr
troceo   con_ruta reranker                                                 
actual   False    False     0.719  0.906  0.786     0.438     0.656   0.529
                  True      0.688  0.938  0.797     0.438     0.812   0.604
         True     False     0.750  0.875  0.802     0.469     0.656   0.555
                  True      0.688  0.938  0.812     0.438     0.781   0.599
limitado False    False     0.781  0.906  0.828     0.344     0.719   0.492
                  True      0.594  0.906  0.727     0.312     0.781   0.526
         True     False     0.750  0.938  0.826     0.438     0.719   0.560
                  True      0.625  0.906  0.755     0.312     0.781   0.531

## 10. B01 y B08

Rango por evidencia de B01 y B08 en las 8 configuraciones y, para cada troceo, el chunk que contiene la evidencia: su tamaño y la posición de la subcadena dentro del chunk.

In [9]:
for id_ in ("B01", "B08"):
    print(id_, "rango por fuente y por evidencia (NaN = fuera del top 4):")
    print(df[df["id"] == id_][CLAVE + ["rango", "rango_evidencia"]].to_string(index=False))

for id_ in ("B01", "B08"):
    for troceo, chunks in chunks_por_troceo.items():
        for fuente, subcadenas in evidencias[id_].items():
            for c in chunks:
                if c["fuente"] != fuente:
                    continue
                for s in subcadenas:
                    pos = c["texto"].find(s)
                    if pos >= 0:
                        print(f"{id_} {troceo}: chunk {c['id']} de {fuente}, {len(c['texto'])} caracteres, "
                              f"evidencia {s[:40]!r} en la posición {pos}")

B01 rango por fuente y por evidencia (NaN = fuera del top 4):
  troceo  con_ruta  reranker  rango  rango_evidencia
  actual     False     False    1.0              NaN
  actual     False      True    1.0              NaN
  actual      True     False    1.0              4.0
  actual      True      True    1.0              NaN
limitado     False     False    1.0              2.0
limitado     False      True    1.0              1.0
limitado      True     False    1.0              1.0
limitado      True      True    1.0              1.0
B08 rango por fuente y por evidencia (NaN = fuera del top 4):
  troceo  con_ruta  reranker  rango  rango_evidencia
  actual     False     False    3.0              NaN
  actual     False      True    3.0              NaN
  actual      True     False    NaN              NaN
  actual      True      True    2.0              NaN
limitado     False     False    1.0              1.0
limitado     False      True    3.0              3.0
limitado      True     False

## 11. Tabla resumen y CSV

`data/resultados/troceo_limitado.csv` tiene una fila por configuración. `data/resultados/troceo_limitado_por_pregunta.csv` tiene una fila por pregunta y configuración con el rango por fuente y por evidencia (vacío si la pregunta no tiene fuentes, categoría E).

In [10]:
filas = []
for config in CONFIGURACIONES:
    nombre = nombre_coleccion(config)
    m = por_config.loc[(config["troceo"], config["con_ruta"], config["reranker"])]
    sub = df[(df["troceo"] == config["troceo"]) & (df["con_ruta"] == config["con_ruta"])
             & (df["reranker"] == config["reranker"])]
    est = estadisticas[config["troceo"]]
    filas.append({
        "troceo": config["troceo"], "con_ruta": config["con_ruta"], "reranker": config["reranker"],
        "coleccion": nombre, "chunks": est["chunks"],
        "chunks por encima del máximo": est["encima"],
        "chunks por encima del máximo que no son un bloque de código": est["encima_no_codigo"],
        "tamaño medio (caracteres)": est["medio"], "tamaño máximo (caracteres)": est["maximo"],
        "evidencias partidas": len(partidas[config["troceo"]]),
        "hit@1": m["hit@1"], f"hit@{K}": m[f"hit@{K}"], "MRR": m["mrr"],
        "ev_hit@1": m["ev_hit@1"], f"ev_hit@{K}": m[f"ev_hit@{K}"], "ev_MRR": m["ev_mrr"],
        "latencia de búsqueda (ms)": sub["busqueda_ms"].mean(),
        "latencia de reranking (ms)": sub["rerank_ms"].mean(),
        "latencia total (ms)": sub["total_ms"].mean(),
        "indexación (s)": t_indexacion[nombre],
    })
resumen = pd.DataFrame(filas)
assert not resumen.isna().any().any(), "El CSV tendría celdas vacías"
RUTA_CSV.parent.mkdir(parents=True, exist_ok=True)
resumen.to_csv(RUTA_CSV, index=False)
print(f"Guardado en {RUTA_CSV.relative_to(RAIZ)}")

preguntas_csv = df[CLAVE + ["coleccion", "id", "categoria", "rango", "rango_evidencia"]].rename(
    columns={"rango": "rango_fuente"}
)
preguntas_csv.to_csv(RUTA_CSV_PREGUNTAS, index=False)
print(f"Guardado en {RUTA_CSV_PREGUNTAS.relative_to(RAIZ)} ({len(preguntas_csv)} filas)")
resumen.round(3)

Guardado en data\resultados\troceo_limitado.csv
Guardado en data\resultados\troceo_limitado_por_pregunta.csv (280 filas)


,troceo,con_ruta,reranker,coleccion,chunks,chunks por encima del máximo,chunks por encima del máximo que no son un bloque de código,tamaño medio (caracteres),tamaño máximo (caracteres),evidencias partidas,hit@1,hit@4,MRR,ev_hit@1,ev_hit@4,ev_MRR,latencia de búsqueda (ms),latencia de reranking (ms),latencia total (ms),indexación (s)
0,actual,False,False,pydocs_nomic_est1500_m200_sinruta,7955,773,709,1203.008,203072,0,0.719,0.906,0.786,0.438,0.656,0.529,84.732,0.000,84.732,228.249
1,actual,False,True,pydocs_nomic_est1500_m200_sinruta,7955,773,709,1203.008,203072,0,0.688,0.938,0.797,0.438,0.812,0.604,84.732,1111.200,1195.933,228.249
2,actual,True,False,pydocs_nomic_est1500_m200_ruta,7955,773,709,1203.008,203072,0,0.750,0.875,0.802,0.469,0.656,0.555,85.641,0.000,85.641,129.770
3,actual,True,True,pydocs_nomic_est1500_m200_ruta,7955,773,709,1203.008,203072,0,0.688,0.938,0.812,0.438,0.781,0.599,85.641,1095.264,1180.905,129.770
4,limitado,False,False,pydocs_nomic_lim1500_m200_sinruta,9143,69,0,1079.120,22273,0,0.781,0.906,0.828,0.344,0.719,0.492,79.023,0.000,79.023,146.293
5,limitado,False,True,pydocs_nomic_lim1500_m200_sinruta,9143,69,0,1079.120,22273,0,0.594,0.906,0.727,0.312,0.781,0.526,79.023,956.713,1035.735,146.293
6,limitado,True,False,pydocs_nomic_lim1500_m200_ruta,9143,69,0,1079.120,22273,0,0.750,0.938,0.826,0.438,0.719,0.560,78.478,0.000,78.478,146.857
7,limitado,True,True,pydocs_nomic_lim1500_m200_ruta,9143,69,0,1079.120,22273,0,0.625,0.906,0.755,0.312,0.781,0.531,78.478,922.257,1000.735,146.857


## 12. Decisión (RF-14)

Un acierto es una pregunta A-D con rango de 1 a 4. Se cuentan, con reranker: `ev` (aciertos por evidencia) y `hit` (aciertos por fuente). El troceo nuevo sin ruta se adopta si `ev` y `hit` no bajan frente al troceo actual sin ruta. El troceo nuevo con ruta se adopta en su lugar si `ev` y `hit` no bajan frente al troceo actual sin ruta y además suma al menos 2 aciertos `ev` más que el nuevo sin ruta. Si ninguna de las dos se cumple, sigue la colección actual sin ruta. Esta celda no toca `scripts/consultar_rag.py`.

In [11]:
def aciertos(troceo: str, con_ruta: bool) -> tuple[int, int]:
    sub = df_abcd[(df_abcd["troceo"] == troceo) & (df_abcd["con_ruta"] == con_ruta) & (df_abcd["reranker"])]
    ev = int(sub["rango_evidencia"].between(1, K).sum())
    hit = int(sub["rango"].between(1, K).sum())
    return ev, hit


ev_a, hit_a = aciertos("actual", False)
ev_ls, hit_ls = aciertos("limitado", False)
ev_lr, hit_lr = aciertos("limitado", True)
print(f"actual sin ruta:    ev={ev_a} hit={hit_a}")
print(f"limitado sin ruta:  ev={ev_ls} hit={hit_ls}")
print(f"limitado con ruta:  ev={ev_lr} hit={hit_lr}")

adopta_sin_ruta = ev_ls >= ev_a and hit_ls >= hit_a
adopta_con_ruta = ev_lr >= ev_a and hit_lr >= hit_a and ev_lr >= ev_ls + 2
if adopta_con_ruta:
    decision = f"pydocs_{MODELO}_lim{MAXIMO}_m{MINIMO}_ruta"
elif adopta_sin_ruta:
    decision = f"pydocs_{MODELO}_lim{MAXIMO}_m{MINIMO}_sinruta"
else:
    decision = COLECCION_PROTEGIDA
print("Colección elegida:", decision)

actual sin ruta:    ev=26 hit=30
limitado sin ruta:  ev=25 hit=29
limitado con ruta:  ev=25 hit=29
Colección elegida: pydocs_nomic_est1500_m200_sinruta
